In [6]:
#!/usr/bin/env python3
import warnings

warnings.filterwarnings(
    'ignore',
    category=UserWarning,
    message=".*Overwriting existing videos.*"
)


# Expert performance

In [5]:

"""
Simple inference script that works with your gymnasium version
"""
import gymnasium as gym
from stable_baselines3 import PPO
import numpy as np
import os
from datetime import datetime

print("Loading PPO model...")
model = PPO.load("./logs/ppo/CarRacing-v3_6/best_model.zip")

# Create environment
print("Creating CarRacing-v3 environment...")
env = gym.make("CarRacing-v3", continuous=True, render_mode="rgb_array")

# Apply basic preprocessing
from gymnasium.wrappers import ResizeObservation, GrayscaleObservation

env = ResizeObservation(env, shape=(64, 64))
env = GrayscaleObservation(env, keep_dim=False)

# Manual frame stacking implementation
class SimpleFrameStack(gym.Wrapper):
    def __init__(self, env, n_frames=2):
        super().__init__(env)
        self.n_frames = n_frames
        self.frames = []
        
    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        # Initialize with the same frame repeated
        self.frames = [obs.copy() for _ in range(self.n_frames)]
        return np.stack(self.frames, axis=0), info
        
    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        # Add new frame and remove oldest
        self.frames.append(obs.copy())
        if len(self.frames) > self.n_frames:
            self.frames.pop(0)
        stacked_obs = np.stack(self.frames, axis=0)
        return stacked_obs, reward, terminated, truncated, info

# Apply frame stacking
env = SimpleFrameStack(env, n_frames=2)

print(f"Environment observation space: {env.observation_space}")
print(f"Environment action space: {env.action_space}")

# Wrap with video recording
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
video_folder = f"videos_{timestamp}"
os.makedirs(video_folder, exist_ok=True)

env = gym.wrappers.RecordVideo(
    env, 
    video_folder=video_folder,
    episode_trigger=lambda x: True,
    name_prefix="ppo_carracing"
)

print(f"Video will be saved to: {video_folder}")
print("Starting inference...")
print("Press Ctrl+C to stop")

try:
    total_rewards = []
    episode_data = []
    
    for episode in range(3):
        print(f"\n=== Episode {episode + 1} ===")
        obs, info = env.reset()
        print(f"Observation shape: {obs.shape}")
        
        episode_reward = 0
        step_count = 0
        max_steps = 1000
        
        while step_count < max_steps:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            episode_reward += reward
            step_count += 1
            
            if step_count % 100 == 0:
                print(f"Episode {episode+1}, Step {step_count}, Reward: {episode_reward:.2f}")
            
            if done:
                print(f"Episode ended: terminated={terminated}, truncated={truncated}")
                break
        
        total_rewards.append(episode_reward)
        episode_data.append({
            'episode': episode + 1,
            'reward': episode_reward,
            'steps': step_count
        })
        print(f"Episode {episode+1} completed. Reward: {episode_reward:.2f}, Steps: {step_count}")
    
    print(f"\n=== Final Results ===")
    print(f"Average reward: {np.mean(total_rewards):.2f}")
    print(f"Reward std: {np.std(total_rewards):.2f}")
    print(f"Min reward: {np.min(total_rewards):.2f}")
    print(f"Max reward: {np.max(total_rewards):.2f}")
    
    # Save results
    results_file = f"results_{timestamp}.txt"
    with open(results_file, 'w') as f:
        f.write("=== PPO CarRacing-v3 Results ===\n")
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Average Reward: {np.mean(total_rewards):.2f}\n")
        f.write(f"Reward Std: {np.std(total_rewards):.2f}\n")
        f.write(f"Min Reward: {np.min(total_rewards):.2f}\n")
        f.write(f"Max Reward: {np.max(total_rewards):.2f}\n")
        f.write("\nEpisode Details:\n")
        for data in episode_data:
            f.write(f"Episode {data['episode']}: Reward={data['reward']:.2f}, Steps={data['steps']}\n")
    
    print(f"Results saved to: {results_file}")
    print(f"Videos saved to: {video_folder}")
    
except KeyboardInterrupt:
    print("\nStopping inference...")
except Exception as e:
    print(f"Error: {e}")
finally:
    env.close()


Loading PPO model...
Creating CarRacing-v3 environment...
Environment observation space: Box(0, 255, (64, 64), uint8)
Environment action space: Box([-1.  0.  0.], 1.0, (3,), float32)
Video will be saved to: videos_20251022_012428
Starting inference...
Press Ctrl+C to stop

=== Episode 1 ===
Observation shape: (2, 64, 64)
Episode 1, Step 100, Reward: 40.54
Episode 1, Step 200, Reward: 113.57
Episode 1, Step 300, Reward: 204.66
Episode 1, Step 400, Reward: 310.18
Episode 1, Step 500, Reward: 437.36
Episode 1, Step 600, Reward: 571.77
Episode 1, Step 700, Reward: 713.39
Episode 1, Step 800, Reward: 779.21
Episode 1, Step 900, Reward: 769.21
Episode 1, Step 1000, Reward: 759.21
Episode ended: terminated=False, truncated=True
Episode 1 completed. Reward: 759.21, Steps: 1000

=== Episode 2 ===
Observation shape: (2, 64, 64)
Episode 2, Step 100, Reward: 28.15
Episode 2, Step 200, Reward: 80.82
Episode 2, Step 300, Reward: 144.39
Episode 2, Step 400, Reward: 224.31
Episode 2, Step 500, Reward:

# Behavourial Cloning

In [ ]:
# Method 3: Behavioral Cloning (BC) Model
print("=== Testing Behavioral Cloning Model ===")

# Import functions from BC training script
sys.path.append('./behavioural_cloning')
from train_bc_with_fallcount import StudentFrameStack, preprocess_state_for_student, N_STACKED_FRAMES
# from train_bc import StudentFrameStack, preprocess_state_for_student, N_STACKED_FRAMES

# Load BC model
BC_MODEL_PATH = "./behavioural_cloning/results/bc_student_model.keras"
model = tf.keras.models.load_model(BC_MODEL_PATH)
print("BC model loaded successfully!")

# Create environment (raw, no preprocessing)
env = gym.make("CarRacing-v3", continuous=True, render_mode="rgb_array")

# Setup video recording
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
video_folder = f"bc_videos_{timestamp}"
os.makedirs(video_folder, exist_ok=True)

env = gym.wrappers.RecordVideo(
    env, 
    video_folder=video_folder,
    episode_trigger=lambda x: True,
    name_prefix="bc_model"
)

print(f"Video will be saved to: {video_folder}")
print("Starting BC inference...")

try:
    total_rewards = []
    episode_data = []
    
    for episode in range(3):
        print(f"\n=== Episode {episode + 1} ===")
        
        # Reset environment and frame stacker
        obs_raw, info = env.reset()
        student_stacker = StudentFrameStack(n_frames=N_STACKED_FRAMES)
        obs = student_stacker.reset(obs_raw)
        
        print(f"Raw observation shape: {obs_raw.shape}")
        print(f"Processed observation shape: {obs.shape}")
        
        episode_reward = 0
        step_count = 0
        max_steps = 1000
        
        while step_count < max_steps:
            # Model prediction (add batch dimension)
            action = model.predict(np.expand_dims(obs, axis=0), verbose=0)[0]
            
            # Step environment
            next_obs_raw, reward, terminated, truncated, info = env.step(action)
            next_obs = student_stacker.step(next_obs_raw)
            done = terminated or truncated
            
            episode_reward += reward
            step_count += 1
            
            if step_count % 100 == 0:
                print(f"Episode {episode+1}, Step {step_count}, Reward: {episode_reward:.2f}")
            
            if done:
                print(f"Episode ended: terminated={terminated}, truncated={truncated}")
                break
            
            obs = next_obs
        
        total_rewards.append(episode_reward)
        episode_data.append({
            'episode': episode + 1,
            'reward': episode_reward,
            'steps': step_count
        })
        print(f"Episode {episode+1} completed. Reward: {episode_reward:.2f}, Steps: {step_count}")
    
    print(f"\n=== BC Results ===")
    print(f"Average reward: {np.mean(total_rewards):.2f}")
    print(f"Reward std: {np.std(total_rewards):.2f}")
    print(f"Min reward: {np.min(total_rewards):.2f}")
    print(f"Max reward: {np.max(total_rewards):.2f}")
    
    # Save results
    results_file = f"bc_results_{timestamp}.txt"
    with open(results_file, 'w') as f:
        f.write("=== Behavioral Cloning CarRacing-v3 Results ===\n")
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Model: {BC_MODEL_PATH}\n")
        f.write(f"Average Reward: {np.mean(total_rewards):.2f}\n")
        f.write(f"Reward Std: {np.std(total_rewards):.2f}\n")
        f.write(f"Min Reward: {np.min(total_rewards):.2f}\n")
        f.write(f"Max Reward: {np.max(total_rewards):.2f}\n")
        f.write("\nEpisode Details:\n")
        for data in episode_data:
            f.write(f"Episode {data['episode']}: Reward={data['reward']:.2f}, Steps={data['steps']}\n")
    
    print(f"Results saved to: {results_file}")
    print(f"Videos saved to: {video_folder}")
    
except Exception as e:
    print(f"Error: {e}")
finally:
    env.close()

bc_results = {
    'method': 'Behavioral Cloning',
    'avg_reward': np.mean(total_rewards),
    'std_reward': np.std(total_rewards),
    'min_reward': np.min(total_rewards),
    'max_reward': np.max(total_rewards),
    'episodes': episode_data
}


=== Testing Behavioral Cloning Model ===
BC model loaded successfully!
Video will be saved to: bc_videos_20251022_155303
Starting BC inference...

=== Episode 1 ===
Raw observation shape: (96, 96, 3)
Processed observation shape: (64, 64, 2)
Episode 1, Step 100, Reward: 35.60
Episode 1, Step 200, Reward: 71.21
Episode 1, Step 300, Reward: 126.35
Episode 1, Step 400, Reward: 122.87
Episode 1, Step 500, Reward: 112.87
Episode 1, Step 600, Reward: 102.87
Episode 1, Step 700, Reward: 92.87
Episode 1, Step 800, Reward: 92.64
Episode 1, Step 900, Reward: 115.21
Episode 1, Step 1000, Reward: 105.21
Episode ended: terminated=False, truncated=True
Episode 1 completed. Reward: 105.21, Steps: 1000

=== Episode 2 ===
Raw observation shape: (96, 96, 3)
Processed observation shape: (64, 64, 2)
Episode 2, Step 100, Reward: 23.76
Episode 2, Step 200, Reward: 81.27
Episode 2, Step 300, Reward: 92.36
Episode 2, Step 400, Reward: 82.36
Episode 2, Step 500, Reward: 72.36
Episode 2, Step 600, Reward: 62.36


# Dagger performance

In [ ]:
#!/usr/bin/env python3
"""
Test script for DAgger model inference with video recording
Based on the check-performance.ipynb notebook but adapted for DAgger model
"""
import gymnasium as gym
import numpy as np
import tensorflow as tf
import os
from datetime import datetime

# Import functions from the DAgger training script
import sys
sys.path.append('./dagger_implementations')
from train_dagger_with_fallcount import StudentFrameStack, preprocess_state_for_student, N_STACKED_FRAMES

# Model path
DAGGER_MODEL_PATH = "./dagger_implementations/results/dagger_student_ppo_with_fallcount.keras"

print("Loading DAgger model...")
try:
    model = tf.keras.models.load_model(DAGGER_MODEL_PATH)
    print("DAgger model loaded successfully!")
except Exception as e:
    print(f"Error loading DAgger model: {e}")
    raise e


Loading DAgger model...
DAgger model loaded successfully!


In [17]:
# Create environment (raw, no preprocessing)
print("Creating CarRacing-v3 environment...")
env = gym.make("CarRacing-v3", continuous=True, render_mode="rgb_array")

print(f"Environment observation space: {env.observation_space}")
print(f"Environment action space: {env.action_space}")

# Wrap with video recording
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
video_folder = f"dagger_videos_{timestamp}"
os.makedirs(video_folder, exist_ok=True)

env = gym.wrappers.RecordVideo(
    env, 
    video_folder=video_folder,
    episode_trigger=lambda x: True,
    name_prefix="dagger_carracing"
)

print(f"Video will be saved to: {video_folder}")

try:
    total_rewards = []
    episode_data = []
    
    for episode in range(3):
        print(f"\n=== Episode {episode + 1} ===")
        
        # Reset environment and frame stacker
        obs_raw, info = env.reset()
        student_stacker = StudentFrameStack(n_frames=N_STACKED_FRAMES)
        obs = student_stacker.reset(obs_raw)
        
        print(f"Raw observation shape: {obs_raw.shape}")
        print(f"Processed observation shape: {obs.shape}")
        
        episode_reward = 0
        step_count = 0
        max_steps = 1000
        
        while step_count < max_steps:
            # Model prediction (add batch dimension)
            action = model.predict(np.expand_dims(obs, axis=0), verbose=0)[0]
            
            # Step environment
            next_obs_raw, reward, terminated, truncated, info = env.step(action)
            next_obs = student_stacker.step(next_obs_raw)
            done = terminated or truncated
            
            episode_reward += reward
            step_count += 1
            
            if step_count % 100 == 0:
                print(f"Episode {episode+1}, Step {step_count}, Reward: {episode_reward:.2f}")
            
            if done:
                print(f"Episode ended: terminated={terminated}, truncated={truncated}")
                break
            
            obs = next_obs
        
        total_rewards.append(episode_reward)
        episode_data.append({
            'episode': episode + 1,
            'reward': episode_reward,
            'steps': step_count
        })
        print(f"Episode {episode+1} completed. Reward: {episode_reward:.2f}, Steps: {step_count}")
    
    print(f"\n=== Final Results ===")
    print(f"Average reward: {np.mean(total_rewards):.2f}")
    print(f"Reward std: {np.std(total_rewards):.2f}")
    print(f"Min reward: {np.min(total_rewards):.2f}")
    print(f"Max reward: {np.max(total_rewards):.2f}")
    
    # Save results
    results_file = f"dagger_results_{timestamp}.txt"
    with open(results_file, 'w') as f:
        f.write("=== DAgger CarRacing-v3 Results ===\n")
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Model: {DAGGER_MODEL_PATH}\n")
        f.write(f"Average Reward: {np.mean(total_rewards):.2f}\n")
        f.write(f"Reward Std: {np.std(total_rewards):.2f}\n")
        f.write(f"Min Reward: {np.min(total_rewards):.2f}\n")
        f.write(f"Max Reward: {np.max(total_rewards):.2f}\n")
        f.write("\nEpisode Details:\n")
        for data in episode_data:
            f.write(f"Episode {data['episode']}: Reward={data['reward']:.2f}, Steps={data['steps']}\n")
    
    print(f"Results saved to: {results_file}")
    print(f"Videos saved to: {video_folder}")
    
except KeyboardInterrupt:
    print("\nStopping inference...")
except Exception as e:
    print(f"Error: {e}")
finally:
    env.close()


Creating CarRacing-v3 environment...
Environment observation space: Box(0, 255, (96, 96, 3), uint8)
Environment action space: Box([-1.  0.  0.], 1.0, (3,), float32)
Video will be saved to: dagger_videos_20251023_074929

=== Episode 1 ===
Raw observation shape: (96, 96, 3)
Processed observation shape: (64, 64, 2)
Episode 1, Step 100, Reward: 37.78
Episode 1, Step 200, Reward: 106.28
Episode 1, Step 300, Reward: 188.43
Episode 1, Step 400, Reward: 284.23
Episode 1, Step 500, Reward: 359.56
Episode 1, Step 600, Reward: 387.10
Episode 1, Step 700, Reward: 465.84
Episode 1, Step 800, Reward: 551.40
Episode 1, Step 900, Reward: 647.20
Episode 1, Step 1000, Reward: 640.61
Episode ended: terminated=False, truncated=True
Episode 1 completed. Reward: 640.61, Steps: 1000

=== Episode 2 ===
Raw observation shape: (96, 96, 3)
Processed observation shape: (64, 64, 2)
Episode 2, Step 100, Reward: 41.66
Episode 2, Step 200, Reward: 116.53
Episode 2, Step 300, Reward: 206.16
Episode 2, Step 400, Reward

# SMILe performance (In progress)

In [ ]:

print("=== Testing SMILe Model ===")

# Import functions from SMILe training script
sys.path.append('./smiLe_implementation')
from train_smile import preprocess_state_for_student, N_STUDENT_FRAMES
import collections

# Load SMILe model
SMILE_MODEL_PATH = "./smiLe_implementation/results/student_smile.keras"
model = tf.keras.models.load_model(SMILE_MODEL_PATH)
print("SMILe model loaded successfully!")

# Create environment (raw, no preprocessing)
env = gym.make("CarRacing-v3", continuous=True, render_mode="rgb_array")

# Setup video recording
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
video_folder = f"smile_videos_{timestamp}"
os.makedirs(video_folder, exist_ok=True)

env = gym.wrappers.RecordVideo(
    env, 
    video_folder=video_folder,
    episode_trigger=lambda x: True,
    name_prefix="smile_model"
)

print(f"Video will be saved to: {video_folder}")
print("Starting SMILe inference...")

try:
    total_rewards = []
    episode_data = []
    
    for episode in range(3):
        print(f"\n=== Episode {episode + 1} ===")
        
        # Reset environment and frame stacker
        obs_raw, info = env.reset()
        
        # Initialize student frame stack (SMILe uses collections.deque)
        student_frame_stack = collections.deque(maxlen=N_STUDENT_FRAMES)
        processed = preprocess_state_for_student(obs_raw)
        for _ in range(N_STUDENT_FRAMES):
            student_frame_stack.append(processed)
        
        print(f"Raw observation shape: {obs_raw.shape}")
        print(f"Processed observation shape: {processed.shape}")
        
        episode_reward = 0
        step_count = 0
        max_steps = 1000
        
        while step_count < max_steps:
            # Create stack: (64, 64, N_STUDENT_FRAMES)
            obs = np.stack(student_frame_stack, axis=-1)
            
            # Model prediction (add batch dimension)
            action = model.predict(np.expand_dims(obs, axis=0), verbose=0)[0]
            
            # Step environment
            next_obs_raw, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            
            # Add new preprocessed frame to stack
            student_frame_stack.append(preprocess_state_for_student(next_obs_raw))
            
            episode_reward += reward
            step_count += 1
            
            if step_count % 100 == 0:
                print(f"Episode {episode+1}, Step {step_count}, Reward: {episode_reward:.2f}")
            
            if done:
                print(f"Episode ended: terminated={terminated}, truncated={truncated}")
                break
        
        total_rewards.append(episode_reward)
        episode_data.append({
            'episode': episode + 1,
            'reward': episode_reward,
            'steps': step_count
        })
        print(f"Episode {episode+1} completed. Reward: {episode_reward:.2f}, Steps: {step_count}")
    
    print(f"\n=== SMILe Results ===")
    print(f"Average reward: {np.mean(total_rewards):.2f}")
    print(f"Reward std: {np.std(total_rewards):.2f}")
    print(f"Min reward: {np.min(total_rewards):.2f}")
    print(f"Max reward: {np.max(total_rewards):.2f}")
    
    # Save results
    results_file = f"smile_results_{timestamp}.txt"
    with open(results_file, 'w') as f:
        f.write("=== SMILe CarRacing-v3 Results ===\n")
        f.write(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Model: {SMILE_MODEL_PATH}\n")
        f.write(f"Average Reward: {np.mean(total_rewards):.2f}\n")
        f.write(f"Reward Std: {np.std(total_rewards):.2f}\n")
        f.write(f"Min Reward: {np.min(total_rewards):.2f}\n")
        f.write(f"Max Reward: {np.max(total_rewards):.2f}\n")
        f.write("\nEpisode Details:\n")
        for data in episode_data:
            f.write(f"Episode {data['episode']}: Reward={data['reward']:.2f}, Steps={data['steps']}\n")
    
    print(f"Results saved to: {results_file}")
    print(f"Videos saved to: {video_folder}")
    
except Exception as e:
    print(f"Error: {e}")
finally:
    env.close()

smile_results = {
    'method': 'SMILe',
    'avg_reward': np.mean(total_rewards),
    'std_reward': np.std(total_rewards),
    'min_reward': np.min(total_rewards),
    'max_reward': np.max(total_rewards),
    'episodes': episode_data
}


=== Testing SMILe Model ===
SMILe model loaded successfully!
Video will be saved to: smile_videos_20251022_161346
Starting SMILe inference...

=== Episode 1 ===
Raw observation shape: (96, 96, 3)
Processed observation shape: (64, 64)
Episode 1, Step 100, Reward: 36.01
Episode 1, Step 200, Reward: 99.63
Episode 1, Step 300, Reward: 175.52
Episode 1, Step 400, Reward: 263.68
Episode 1, Step 500, Reward: 361.04
Episode 1, Step 600, Reward: 476.81
Episode 1, Step 700, Reward: 589.51
Episode 1, Step 800, Reward: 702.21
Episode 1, Step 900, Reward: 707.55
Episode 1, Step 1000, Reward: 697.55
Episode ended: terminated=False, truncated=True
Episode 1 completed. Reward: 697.55, Steps: 1000

=== Episode 2 ===
Raw observation shape: (96, 96, 3)
Processed observation shape: (64, 64)
Episode 2, Step 100, Reward: 35.16
Episode 2, Step 200, Reward: 96.13
Episode 2, Step 300, Reward: 163.55
Episode 2, Step 400, Reward: 253.55
Episode 2, Step 500, Reward: 350.00
Episode 2, Step 600, Reward: 362.58
Epis